# 02 · Preparación de datos y modelos supervisados

EP2 · Machine Learning · Waymo Open Dataset v2.0.1 (`lidar_box`)

Este notebook tiene dos partes. La **Parte A** (este paso) deja los datos listos y define cómo vamos a evaluar sin hacer trampa. La **Parte B** (siguiente paso) entrena y compara los modelos.

Las dos tareas:
- **Clasificación:** predecir el tipo de objeto (Vehículo, Señal, Peatón, Ciclista).
- **Regresión:** predecir la distancia entre el vehículo y el objeto, en metros.

In [1]:
import sys
sys.path.append("..")  # permite importar el paquete src/ desde notebooks/

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier, DummyRegressor
from sklearn.model_selection import GroupKFold, StratifiedGroupKFold, cross_validate
from sklearn.pipeline import Pipeline

from src.data_preprocessing import (
    FEATURES_CLASIFICACION,
    FEATURES_REGRESION,
    RANDOM_STATE,
    agregar_features,
    agregar_nombre_tipo,
    cargar_segmentos,
    construir_preprocesador,
    dividir_por_segmentos,
    elegir_segmentos_prueba,
    preparar_clasificacion,
    preparar_regresion,
    submuestrear_frames,
    verificar_sin_fuga,
)

ORDEN_CLASES = ["Vehículo", "Señal", "Peatón", "Ciclista"]

## Parte A · Preparación de los datos

### 1. Carga y variables derivadas

Además de las columnas originales, calculamos cuatro variables que resumen mejor lo que mide el sensor:
- `volumen`: largo × ancho × alto de la caja.
- `rapidez`: velocidad total del objeto (m/s), sin importar la dirección.
- `aceleracion`: aceleración total del objeto.
- `distancia_vehiculo`: distancia en metros entre el vehículo y el objeto.

In [2]:
df = agregar_features(agregar_nombre_tipo(cargar_segmentos("../data/raw/lidar_box")))
print(f"Filas: {len(df):,} | Segmentos: {df['key_segment_context_name'].nunique()}")
df[["volumen", "rapidez", "aceleracion", "distancia_vehiculo"]].describe().round(2)

Filas: 332,741 | Segmentos: 30


,volumen,rapidez,aceleracion,distancia_vehiculo
count,332741.00,332741.00,332741.00,332741.00
mean,9.34,1.00,0.11,40.83
std,14.26,3.05,0.33,19.50
min,0.00,0.00,0.00,2.32
25%,0.11,0.00,0.00,25.08
50%,2.19,0.02,0.00,40.12
75%,15.45,0.14,0.06,56.51
max,251.48,31.58,12.82,85.92


### 2. Reducir frames repetidos

El sensor mide 10 veces por segundo, así que un mismo objeto aparece casi igual en frames seguidos. Nos quedamos con **1 frame de cada 5** dentro de cada segmento. Así el entrenamiento es unas 5 veces más rápido y casi no se pierde información (la pauta pide usar bien los recursos de cómputo). Revisamos que no se pierdan objetos, sobre todo ciclistas.

In [3]:
CADA_N = 5
df_sub = submuestrear_frames(df, cada_n=CADA_N)

comparacion = pd.DataFrame({
    "filas_antes": df["type_nombre"].value_counts().reindex(ORDEN_CLASES),
    "filas_despues": df_sub["type_nombre"].value_counts().reindex(ORDEN_CLASES),
    "objetos_antes": df.groupby("type_nombre")["key_laser_object_id"].nunique().reindex(ORDEN_CLASES),
    "objetos_despues": df_sub.groupby("type_nombre")["key_laser_object_id"].nunique().reindex(ORDEN_CLASES),
})
print(f"Filas: {len(df):,} -> {len(df_sub):,}")
comparacion

Filas: 332,741 -> 67,158


,filas_antes,filas_despues,objetos_antes,objetos_despues
type_nombre,,,,
Vehículo,162697,32846,1639,1626
Señal,96564,19492,976,973
Peatón,71714,14462,586,586
Ciclista,1766,358,17,17


**Hallazgo:** _(¿se perdió algún objeto de alguna clase? ¿cuántos ciclistas quedan?)._

**Hallazgo:** al quedarnos con 1 frame de cada 5, bajamos de 332.741 a 67.158 filas. Se perdieron muy pocos objetos: 13 vehículos y 3 señales, que aparecían en muy pocos frames. No se perdió ningún peatón y los 17 ciclistas siguen estando todos (con 358 filas). Así el entrenamiento será unas 5 veces más rápido sin perder ninguna clase.

### 3. Separar entrenamiento y prueba por segmentos completos

Reservamos cerca del 20 % de los **segmentos** para una prueba final que no se toca hasta el final. Los segmentos se separan completos (nunca se parten), así el modelo no se prueba con objetos que ya vio. La elección usa `StratifiedGroupKFold` para que las clases queden repartidas de forma parecida, y una semilla fija para que sea reproducible.

In [4]:
segmentos_prueba = elegir_segmentos_prueba(df, random_state=RANDOM_STATE)
df_train, df_test = dividir_por_segmentos(df_sub, segmentos_prueba)
verificar_sin_fuga(df_train, df_test)  # falla si algún segmento u objeto está en los dos lados
print("Sin fuga entre entrenamiento y prueba: OK")

def resumen(d):
    return pd.Series({
        "segmentos": d["key_segment_context_name"].nunique(),
        "filas": len(d),
        "objetos": d["key_laser_object_id"].nunique(),
        **{f"objetos {c}": d.loc[d["type_nombre"] == c, "key_laser_object_id"].nunique() for c in ORDEN_CLASES},
    })

pd.DataFrame({"entrenamiento": resumen(df_train), "prueba": resumen(df_test)})

Sin fuga entre entrenamiento y prueba: OK


,entrenamiento,prueba
segmentos,23,7
filas,53672,13486
objetos,2601,601
objetos Vehículo,1317,309
objetos Señal,786,187
objetos Peatón,483,103
objetos Ciclista,15,2


**Hallazgo:** _(cuántos segmentos y cuántos ciclistas quedaron en la prueba; ¿qué tan confiable será medir ciclistas ahí?)._

**Hallazgo:** 23 segmentos quedaron para entrenar y 7 para la prueba final (13.486 filas y 601 objetos). Ningún segmento ni objeto está en los dos lados, y lo comprobamos con código. El punto débil son los ciclistas: de los 17, 15 quedaron en entrenamiento y solo 2 en la prueba. Por eso el resultado de ciclistas en la prueba final se basará en solo 2 objetos y no será concluyente. Para esa clase nos apoyaremos más en la validación cruzada del entrenamiento (15 objetos) y lo diremos explícitamente.

### 4. Variables de entrada de cada tarea

**Importante:** en la regresión **no** usamos `box_center_x/y/z` ni `distancia_vehiculo`, porque la distancia se calcula directamente con esas columnas. Eso sería hacer trampa. El modelo tiene que estimar la distancia solo con lo que mide el LiDAR sobre el objeto: su tipo, su tamaño y cuántos puntos recibió.

In [5]:
print("Clasificación:", FEATURES_CLASIFICACION)
print("Regresión:     ", FEATURES_REGRESION)

X_clf_train, y_clf_train, g_clf_train = preparar_clasificacion(df_train)
X_reg_train, y_reg_train, g_reg_train = preparar_regresion(df_train)
print(f"\nEntrenamiento -> clasificación: {X_clf_train.shape} | regresión: {X_reg_train.shape}")

Clasificación: ['box_size_x', 'box_size_y', 'box_size_z', 'volumen', 'rapidez', 'aceleracion', 'num_lidar_points_in_box', 'num_top_lidar_points_in_box', 'distancia_vehiculo']
Regresión:      ['box_size_x', 'box_size_y', 'box_size_z', 'volumen', 'num_lidar_points_in_box', 'num_top_lidar_points_in_box', 'type']

Entrenamiento -> clasificación: (53672, 9) | regresión: (53672, 7)


### 5. Transformación de variables dentro de un pipeline

Las variables se transforman dentro de un `Pipeline` de Scikit-learn:
- Los conteos de puntos LiDAR son muy desiguales (de 0 a miles), así que se les aplica `log(1 + x)`.
- Todas las variables numéricas se escalan (media 0, desviación 1), lo que necesitan los modelos lineales y KNN.
- En regresión, el tipo de objeto se convierte en columnas 0/1 (One-Hot).

Al ir dentro del pipeline, el escalado se calcula **solo con los datos de entrenamiento de cada fold**, así no se filtra información de los datos de validación.

In [6]:
prep_clf = construir_preprocesador("clasificacion").fit(X_clf_train)
prep_reg = construir_preprocesador("regresion").fit(X_reg_train)
print("Clasificación ->", list(prep_clf.get_feature_names_out()))
print("Regresión     ->", list(prep_reg.get_feature_names_out()))

Clasificación -> ['puntos_log__num_lidar_points_in_box', 'puntos_log__num_top_lidar_points_in_box', 'numericas__box_size_x', 'numericas__box_size_y', 'numericas__box_size_z', 'numericas__volumen', 'numericas__rapidez', 'numericas__aceleracion', 'numericas__distancia_vehiculo']
Regresión     -> ['puntos_log__num_lidar_points_in_box', 'puntos_log__num_top_lidar_points_in_box', 'numericas__box_size_x', 'numericas__box_size_y', 'numericas__box_size_z', 'numericas__volumen', 'tipo__type_1', 'tipo__type_2', 'tipo__type_3', 'tipo__type_4']


### 6. Validación cruzada agrupada por segmento

Con los datos de entrenamiento se hace validación cruzada de 5 folds, **sin partir ningún segmento**:
- **Clasificación:** `StratifiedGroupKFold`, que además intenta repartir las clases parejo.
- **Regresión:** `GroupKFold`.

Veamos qué le toca a cada fold, sobre todo en la clase Ciclista.

In [7]:
cv_clf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_reg = GroupKFold(n_splits=5)

filas = []
for k, (_, idx_val) in enumerate(cv_clf.split(X_clf_train, y_clf_train, g_clf_train), start=1):
    val = y_clf_train.iloc[idx_val]
    filas.append({"fold": k,
                  "segmentos": g_clf_train.iloc[idx_val].nunique(),
                  "filas": len(idx_val),
                  **{c: int((val == c).sum()) for c in ORDEN_CLASES}})
print("Clasificación: filas de validación por fold y clase")
display(pd.DataFrame(filas).set_index("fold"))

filas = []
for k, (_, idx_val) in enumerate(cv_reg.split(X_reg_train, y_reg_train, g_reg_train), start=1):
    filas.append({"fold": k, "segmentos": g_reg_train.iloc[idx_val].nunique(), "filas": len(idx_val)})
print("Regresión: filas de validación por fold")
display(pd.DataFrame(filas).set_index("fold"))

Clasificación: filas de validación por fold y clase


,segmentos,filas,Vehículo,Señal,Peatón,Ciclista
fold,,,,,,
1,5,12684,5267,2987,4390,40
2,5,9702,5749,3057,797,99
3,4,9215,4766,3452,957,40
4,3,11016,4681,3229,3060,46
5,6,11055,6093,2972,1920,70


Regresión: filas de validación por fold


,segmentos,filas
fold,,
1,5,11021
2,5,10883
3,4,10422
4,5,10954
5,4,10392


**Hallazgo:** _(¿algún fold se queda sin ciclistas o con muy pocos? ¿qué implica para el recall de esa clase?)._

**Hallazgo:** ningún fold se quedó sin ciclistas: cada uno tiene entre 40 y 99 filas de esa clase. Pero esas filas vienen de muy pocos objetos (15 ciclistas repartidos en 5 folds, unos 3 por fold en promedio), así que el recall de ciclistas puede variar mucho de un fold a otro. Los peatones también están desparejos: van de 797 a 4.390 filas por fold, porque hay segmentos con muchísimos peatones y otros con casi ninguno. En regresión los folds están bien balanceados (entre 10.392 y 11.021 filas). Por eso siempre reportaremos el promedio y cuánto varía entre folds.

### 7. Línea base: qué resultado se obtiene sin aprender nada

Antes de entrenar modelos, medimos modelos "tontos". Son el piso que cualquier modelo real debe superar:
- Clasificación: siempre responder la clase más frecuente.
- Regresión: siempre responder la distancia promedio.

In [8]:
def resumir(res, metricas):
    tabla = pd.DataFrame({m: [res[f"test_{m}"].mean(), res[f"test_{m}"].std()] for m in metricas},
                         index=["media", "desv. entre folds"])
    return tabla.round(3)

base_clf = Pipeline([("prep", construir_preprocesador("clasificacion")),
                     ("modelo", DummyClassifier(strategy="most_frequent"))])
res_clf = cross_validate(base_clf, X_clf_train, y_clf_train, groups=g_clf_train, cv=cv_clf,
                         scoring=["accuracy", "balanced_accuracy", "f1_macro"])
print("Línea base de clasificación (siempre la clase más frecuente)")
display(resumir(res_clf, ["accuracy", "balanced_accuracy", "f1_macro"]))

base_reg = Pipeline([("prep", construir_preprocesador("regresion")),
                     ("modelo", DummyRegressor(strategy="mean"))])
res_reg = cross_validate(base_reg, X_reg_train, y_reg_train, groups=g_reg_train, cv=cv_reg,
                         scoring=["r2", "neg_root_mean_squared_error", "neg_mean_absolute_error"])
res_reg["test_rmse"] = -res_reg["test_neg_root_mean_squared_error"]
res_reg["test_mae"] = -res_reg["test_neg_mean_absolute_error"]
print("Línea base de regresión (siempre la distancia promedio)")
display(resumir(res_reg, ["r2", "rmse", "mae"]))

Línea base de clasificación (siempre la clase más frecuente)


,accuracy,balanced_accuracy,f1_macro
media,0.50,0.25,0.166
desv. entre folds,0.07,0.00,0.016


Línea base de regresión (siempre la distancia promedio)


,r2,rmse,mae
media,-0.013,19.549,16.567
desv. entre folds,0.014,0.544,0.577


**Hallazgo:** _(¿qué accuracy logra el modelo que siempre dice la clase más frecuente y qué dice su balanced accuracy? ¿por qué sirve tener las dos métricas?)._

**Hallazgo:** el modelo que siempre responde "Vehículo" saca un accuracy de 0,50, que parece aceptable. Pero su balanced accuracy es 0,25, justo lo esperable de un modelo sin ninguna capacidad (acierta 1 de cada 4 clases), y su F1 macro es solo 0,17. Por eso usamos las dos métricas: el accuracy premia a la clase grande y puede engañar, mientras que balanced accuracy y F1 macro tratan igual a todas las clases. En regresión, predecir siempre la distancia promedio da R² ≈ 0 (no explica nada), un error típico (RMSE) de 19,5 m y un error promedio (MAE) de 16,6 m. Esos 19,5 m coinciden con la desviación estándar de la distancia que vimos en el EDA, lo que confirma que es la referencia de "no aprender nada". Estos son los números que los modelos reales tienen que superar.

## Decisiones de la Parte A

- **Variables:** para clasificar usamos tamaño, volumen, rapidez, aceleración, puntos LiDAR y distancia. Para la regresión usamos solo tipo, tamaño y puntos LiDAR, sin las coordenadas, porque con ellas la distancia se calcula directo.
- **Menos frames:** 1 de cada 5, para entrenar unas 5 veces más rápido. Solo se perdieron 16 objetos (13 vehículos y 3 señales), ninguno de las clases raras.
- **Prueba final:** 7 de 30 segmentos reservados, completos y sin objetos compartidos con el entrenamiento. Los ciclistas quedan poco representados en la prueba (2 objetos), así que su resultado final será solo orientativo.
- **Validación:** cruzada de 5 folds agrupada por segmento (StratifiedGroupKFold en clasificación, GroupKFold en regresión), reportando promedio y variación entre folds.
- **Líneas base a superar:** clasificación con balanced accuracy 0,25 y F1 macro 0,17; regresión con R² ≈ 0 y RMSE de 19,5 m.